In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # disable GPU devices
os.environ["TFDS_DATA_DIR"] = os.path.expanduser("~/tensorflow_datasets")  # default location of tfds database
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers, models, metrics, losses
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

import tensorflow as tf
import tensorflow_datasets as tfds

import numpy as np

import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.manifold import TSNE

# Turn off logging for TF
import logging
logging.disable(logging.WARNING)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"
tf.get_logger().setLevel(logging.ERROR)

from dpmhm.datasets import preprocessing, feature, utils, transformer

ds_all, ds_info = tfds.load(
    'CWRU',
    with_info=True,
)

ds0 = ds_all['train']
ds0.element_spec

Load registered datasets (if not registered, run creation_metadataset.ipynb)

In [ ]:
outdir = Path('/volatile/home/bm279471/tmp/meta_dataset')
# outdir = Path('/volatile/home/bm279471/tmp/meta_dataset_full_bandwidth')
# outdir = Path('/volatile/home/bm279471/tmp/few_shot_cwru')
os.makedirs(outdir, exist_ok=True)

In [ ]:
ds_train = tf.data.Dataset.load(str(outdir/'ds_train'))
ds_val = tf.data.Dataset.load(str(outdir/'ds_val'))
ds_train_ft = tf.data.Dataset.load(str(outdir/'ds_train_ft'))
ds_val_ft = tf.data.Dataset.load(str(outdir/'ds_val_ft'))
ds_test_ft = tf.data.Dataset.load(str(outdir/'ds_test_ft'))

with open(outdir/'lb1.json', 'r') as fp:
    lb1 = list(json.load(fp))
with open(outdir/'lb2.json', 'r') as fp:
    lb2 = list(json.load(fp))
with open(outdir/'lb3.json', 'r') as fp:
    lb3 = list(json.load(fp))

In [ ]:
batch_size = 32
ds_test_size = utils.get_dataset_size(ds_train_ft)+utils.get_dataset_size(ds_val_ft)+utils.get_dataset_size(ds_test_ft)
ds_train_size = utils.get_dataset_size(ds_train)+utils.get_dataset_size(ds_val)
n_embedding  = 128 
kernel_size = (3,3)
projection_dim = 128
nb_classes=len(lb1)

In [ ]:
ds_train_clr = ds_train.map(lambda x,l:(x,x)).shuffle(ds_train_size, reshuffle_each_iteration=False).cache().batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
ds_val_clr = ds_val.cache().batch(batch_size,drop_remainder=True)
ds_train_ft= ds_train_ft.shuffle(ds_test_size, reshuffle_each_iteration=False).cache().batch(batch_size,drop_remainder=True).prefetch(tf.data.AUTOTUNE)
ds_val_ft = ds_val_ft.cache().batch(batch_size,drop_remainder=True)
ds_test_ft=ds_test_ft.cache().batch(1)

eles = list(ds_train.take(1).as_numpy_iterator())
input_shape = eles[0][0].shape

In [ ]:
tf.config.run_functions_eagerly(True)

@tf.keras.utils.register_keras_serializable()
class AddPositionEmbs(layers.Layer):
    """Adds (optionally learned) positional embeddings to the inputs."""

    def build(self, input_shape):
        assert (
            len(input_shape) == 3
        ), f"Number of dimensions should be 3, got {len(input_shape)}"
        self.pe = tf.Variable(
            name="pos_embedding",
            initial_value=tf.random_normal_initializer(stddev=0.06)(
                shape=(1, input_shape[1], input_shape[2])
            ),
            dtype="float32",
            trainable=True,
        )

    def call(self, inputs):
        return inputs + tf.cast(self.pe, dtype=inputs.dtype)

    def get_config(self):
        config = super().get_config()
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

@tf.keras.utils.register_keras_serializable()
class MultiHeadSelfAttention(layers.Layer):
    def __init__(self, *args, num_heads, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_heads = num_heads

    def build(self, input_shape):
        embedding_dim = input_shape[-1]
        num_heads = self.num_heads
        if embedding_dim % num_heads != 0:
            raise ValueError(
                f"embedding dimension = {embedding_dim} should be divisible by number of heads = {num_heads}"
            )
        self.embedding_dim = embedding_dim
        self.projection_dim = embedding_dim // num_heads
        self.query_dense = layers.Dense(embedding_dim, name="query")
        self.key_dense = layers.Dense(embedding_dim, name="key")
        self.value_dense = layers.Dense(embedding_dim, name="value")
        self.combine_heads = layers.Dense(embedding_dim, name="out")

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], score.dtype)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)
        query = self.separate_heads(query, batch_size)
        key = self.separate_heads(key, batch_size)
        value = self.separate_heads(value, batch_size)

        attention, weights = self.attention(query, key, value)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embedding_dim))
        output = self.combine_heads(concat_attention)
        return output, weights

    def get_config(self):
        config = super().get_config()
        config.update({"num_heads": self.num_heads})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)


@tf.keras.utils.register_keras_serializable()
class TransformerBlock(layers.Layer):
    """Implements a Transformer block."""

    def __init__(self, *args, num_heads, mlp_dim, dropout, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_heads = num_heads
        self.mlp_dim = mlp_dim
        self.dropout = dropout

    def build(self, input_shape):
        self.att = MultiHeadSelfAttention(
            num_heads=self.num_heads,
            name="MultiHeadDotProductAttention_1",
        )
        self.mlpblock = keras.Sequential(
            [
                layers.Dense(
                    self.mlp_dim,
                    activation="linear",
                    name=f"{self.name}_Dense_0",
                ),
                layers.Lambda(
                    lambda x: keras.activations.gelu(x, approximate=False)
                )
                if hasattr(keras.activations, "gelu")
                else layers.Lambda(
                    lambda x: tf.nn.gelu(x, approximate=False)
                ),
                layers.Dropout(self.dropout),
                layers.Dense(input_shape[-1], name=f"{self.name}_Dense_1"),
                layers.Dropout(self.dropout),
            ],
            name="MlpBlock_3",
        )
        self.layernorm1 = layers.LayerNormalization(
            epsilon=1e-6, name="LayerNorm_0"
        )
        self.layernorm2 = layers.LayerNormalization(
            epsilon=1e-6, name="LayerNorm_2"
        )
        self.dropout_layer = layers.Dropout(self.dropout)

    def call(self, inputs, training):
        x = self.layernorm1(inputs)
        x, weights = self.att(x)
        x = self.dropout_layer(x, training=training)
        x = x + inputs
        y = self.layernorm2(x)
        y = self.mlpblock(y)
        return x + y, weights

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "num_heads": self.num_heads,
                "mlp_dim": self.mlp_dim,
                "dropout": self.dropout,
            }
        )
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

@tf.function()
def SampleMaskIndex(seq_len, N, C):
        I = set()
        while len(I) < N:
            i = np.random.randint(0, seq_len)  # uniform distribution over {1, N}
            Ic = set(range(max(0, i-C+1), min(seq_len, i+C)))
            I = I.union(Ic)
        I = list(I)[:N]  # guarantee to mask exactly N patches
        return I

@tf.keras.utils.register_keras_serializable()
class Mask(layers.Layer):
    def __init__(self, mask_ratio=0.15, batch_size=32, embedding_dim=128, seq_len=16, cluster_factor=3, **kwargs):
        super(Mask, self).__init__(**kwargs)
        self.mask_ratio = mask_ratio
        self.batch_size=batch_size
        self.embedding_dim=embedding_dim
        self.seq_len=seq_len
        self.cluster_factor=cluster_factor
        self.learnable_mask = self.add_weight(
            shape=(self.batch_size, seq_len, embedding_dim),
            initializer="glorot_uniform",  
            trainable=True,
            name="learnable_mask",
        )

    def call(self, y):
        N = tf.cast(self.seq_len * self.mask_ratio, tf.int32)

        masked_indices = SampleMaskIndex(self.seq_len, N, self.cluster_factor)
        # Create the mask Tensor and the masked embeddings
        mask = tf.ones((self.batch_size, self.seq_len, self.embedding_dim))

        indices = tf.constant([[i, j] for i in range(self.batch_size) for j in masked_indices])
        updates = tf.zeros((len(indices), self.embedding_dim))

        mask = tf.tensor_scatter_nd_update(mask, indices, updates)
        y_masked = y * mask + (tf.ones((self.batch_size, self.seq_len, self.embedding_dim)) - mask) * self.learnable_mask

        # Create a list of candidates for each masked patch
        candidate_patches = []
        for i in range(self.batch_size):
            batch_candidates=[]
            for j, idx in enumerate(masked_indices):
                # for each masked patch, we introduce the masked patch and then the rest of the masked patches
                patches = []
                patches.append(y[i, idx, :])  # correct index
                for masked_idx in masked_indices[:j] + masked_indices[j+1:]:
                    patches.append(y[i, masked_idx, :])
                batch_candidates.append(patches)
            candidate_patches.append(batch_candidates)

        return y_masked, mask, candidate_patches, masked_indices

@tf.keras.utils.register_keras_serializable()
class SSASTModel(models.Model):
    def __init__(self, image_size=(64,64), patch_size=16, num_layers=3, embedding_dim=128, num_heads=4, mlp_dim=128, dropout=0.1, mask_ratio=0.2, batch_size=32, Lambda=10, cluster_factor=3, **kwargs):
        super(SSASTModel, self).__init__(**kwargs)
        self.seq_len=(image_size[0]//patch_size)*(image_size[1]//patch_size)
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_layers = num_layers
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.mlp_dim = mlp_dim
        self.dropout = dropout
        self.batch_size=batch_size
        self.Lambda=Lambda
        self.mask_ratio=mask_ratio

        self.embedding=layers.Conv2D(
            filters=self.embedding_dim,
            kernel_size=self.patch_size,
            strides=self.patch_size,
            padding="valid",
            name="embedding",
        )
        self.reshape =layers.Reshape((self.seq_len, self.embedding_dim))
        self.position_embedding=AddPositionEmbs(name="Transformer_posembed_input")
        self.mask_layer = Mask(mask_ratio=mask_ratio, batch_size=batch_size, embedding_dim=embedding_dim, seq_len=self.seq_len, cluster_factor=cluster_factor)
        self.transformer=[
            TransformerBlock(
                num_heads=self.num_heads,
                mlp_dim=self.mlp_dim,
                dropout=self.dropout,
                name=f"Transformer_encoderblock_{n}",
            ) for n in range(self.num_layers)
        ]
        self.transformer_norm = layers.LayerNormalization(
            epsilon=1e-6, name="Transformer_encoder_norm"
        )
        self.embedding_inverse = layers.Conv2DTranspose(
            filters=3,
            kernel_size=self.patch_size,
            strides=self.patch_size,
            padding="valid",
            name="embedding_inverse",
        )
        self.reshape_inverse = layers.Reshape((self.image_size[0] // self.patch_size, self.image_size[1] // self.patch_size, self.embedding_dim))

    def reconstruction(self, O, mask):
        '''Reconstructs the original masked patches from the transformer output and computes it to the spectrogram shape'''
        O = (tf.ones((O.shape[0], O.shape[1], O.shape[2])) - mask) * O
        return self.embedding_inverse(self.reshape_inverse(O))
    
    def classification(self, O, mask, candidate_patches, masked_indices):
        """For each masked patch, selects the correct patch among the candidates patches"""
        masked_patches = (tf.ones((mask.shape[0], mask.shape[1], mask.shape[2])) - mask) * O
        classifications = []
        for i, elem in enumerate(masked_patches):
            patch_classifications = []
            for idx, j in enumerate(masked_indices):
                patch = elem[j]
                candidate_similarities = []
                for candidate in candidate_patches[i][idx]:
                    similarity = tf.reduce_mean(patch * candidate, axis=-1)
                    candidate_similarities.append(similarity)
                patch_classification = tf.stack(candidate_similarities)
                patch_classifications.append(patch_classification)
            classifications.append(patch_classifications)
        return classifications

    def convert_list_to_tensor(self, L, dtype=tf.float32):
        max_len = max(len(inner_list) for inner_list in L)
        padded_lists = []
        for inner_list in L:
            padded_list = [tf.convert_to_tensor(item, dtype=dtype) for item in inner_list]
            padded_list += [tf.zeros_like(inner_list[0], dtype=tf.float32)] * (max_len - len(inner_list))
            padded_lists.append(tf.stack(padded_list))
        return tf.stack(padded_lists)

    def call(self, x):
        E = self.embedding(x)
        E = self.reshape(E)
        
        E_mask, mask, candidate_patches, masked_indices = self.mask_layer.call(E)

        O = self.position_embedding(E_mask)
        for n in range(self.num_layers):
            O, _ = self.transformer[n](O, training=True)
        O = self.transformer_norm(O)

        r_i = self.reconstruction(O, mask)
        c_i = self.classification(O, mask, candidate_patches, masked_indices)

        r_i_true = (tf.ones((self.batch_size, E.shape[1], self.embedding_dim)) - mask) * E
        r_i_true = self.embedding_inverse(self.reshape_inverse(r_i_true))
        c_i_true = [[0 for _ in range(len(masked_indices))] for _ in range(self.batch_size)]  # Set all true indices to 0

        c_i = self.convert_list_to_tensor(c_i)
        c_i_true = tf.convert_to_tensor(c_i_true, dtype=tf.int64)

        c_i = tf.reshape(c_i, [-1, int(self.mask_ratio*self.seq_len)])
        c_i_true = tf.reshape(c_i_true, [-1])

        return r_i, r_i_true, c_i, c_i_true

    def compile(self, optimizer, **kwargs):
        super(SSASTModel, self).compile(**kwargs)
        self.optimizer = optimizer
        self.loss_tracker = metrics.Mean(name="loss")
        self.reconstruction_loss_tracker = metrics.Mean(name="reconstruction_loss")
        self.classification_loss_tracker = metrics.Mean(name="classification_loss")

    @property
    def metrics(self):
        return [self.loss_tracker, self.reconstruction_loss_tracker, self.classification_loss_tracker]

    def train_step(self, data):
        x, _ = data
        with tf.GradientTape() as tape:
            r_i, r_i_true, c_i, c_i_true = self(x, training=True)
            reconstruction_loss = self.Lambda*losses.mean_squared_error(r_i_true, r_i)
            classification_loss = losses.sparse_categorical_crossentropy(c_i_true, c_i, from_logits=True)
            total_loss = tf.reduce_mean(reconstruction_loss) + tf.reduce_mean(classification_loss)

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        self.loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.classification_loss_tracker.update_state(classification_loss)

        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, _ = data

        r_i, r_i_true, c_i, c_i_true = self(x, training=False)
        reconstruction_loss = self.Lambda * losses.mean_squared_error(r_i_true, r_i)
        classification_loss = losses.sparse_categorical_crossentropy(c_i_true, c_i, from_logits=True)
        total_loss = tf.reduce_mean(reconstruction_loss) + tf.reduce_mean(classification_loss)

        self.loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.classification_loss_tracker.update_state(classification_loss)
        return {m.name: m.result() for m in self.metrics}

ssast_model = SSASTModel(image_size=(input_shape[0], input_shape[1]), Lambda=10)
test_input = tf.random.normal((32, input_shape[0], input_shape[1], 3))
test_output = ssast_model(test_input)
print(len(test_output), [o.shape for o in test_output])

In [ ]:
ssast_model.compile(
    optimizer=keras.optimizers.Adam()
)

history = ssast_model.fit(
    ds_train_clr.repeat(),
    validation_data=ds_val_clr,
    epochs=5,
    steps_per_epoch=(int(0.8*ds_train_size)//batch_size),
)

In [ ]:
ssast_model.save_weights('ssast_meta_dataset.weights.h5')
ssast_model.load_weights('ssast_meta_dataset.weights.h5')

In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']
reconstruction_loss = history.history['reconstruction_loss']
val_reconstruction_loss = history.history['val_reconstruction_loss']
classification_loss = history.history['classification_loss']
val_classification_loss = history.history['val_classification_loss']

# Créer une figure et des axes
fig, ax = plt.subplots(figsize=(10, 6))

# Plot loss et val_loss
epochs = np.arange(1, len(loss) + 1)
ax.plot(epochs, loss, label='Training Loss', marker='o', linestyle='-', color='r')
ax.plot(epochs, val_loss, label='Validation Loss', marker='o', linestyle='--', color='r')


ax.plot(epochs, reconstruction_loss, label='Training Reconstruction Loss', marker='o', linestyle='-', color='b')
ax.plot(epochs, val_reconstruction_loss, label='Validation Reconstruction Loss', marker='o', linestyle='--', color='b')

ax.plot(epochs, classification_loss, label='Training Classification Loss', marker='o', linestyle='-', color='g')
ax.plot(epochs, val_classification_loss, label='Validation Classification Loss', marker='o', linestyle='--', color='g')

# Configurer les labels et la légende
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.set_title('Training and Validation Losses')
ax.legend()

# Afficher le graphique
plt.tight_layout()
plt.show()

In [ ]:
@tf.keras.utils.register_keras_serializable()
class ClassificationModel(models.Model):
    def __init__(self, model, **kwargs):
        super(ClassificationModel, self).__init__(**kwargs)
        self.num_layers = model.num_layers

        self.embedding = model.embedding
        self.reshape = model.reshape
        self.position_embedding = model.position_embedding
        self.transformer = model.transformer
        self.transformer_norm = model.transformer_norm
        self.meanpooling = layers.GlobalAveragePooling1D(name="MeanPooling")
        self.classification_layer = layers.Dense(nb_classes, name="Classification_head")

    def _set_non_trainable_layers(self):
        for layer in [self.embedding, self.reshape, self.position_embedding, self.transformer_norm]:
            layer.trainable = False
        for transformer in self.transformer:
            transformer.trainable = False
            transformer.att.query_dense.trainable = False
            transformer.att.key_dense.trainable = False
            transformer.att.value_dense.trainable = False
            transformer.att.combine_heads.trainable = False
            transformer.mlpblock.trainable = False
            transformer.layernorm1.trainable = False
            transformer.layernorm2.trainable = False
            transformer.dropout_layer.trainable = False

    def _set_trainable_layers(self):
        for layer in [self.embedding, self.reshape, self.position_embedding, self.transformer_norm]:
            layer.trainable = True
        for transformer in self.transformer:
            transformer.trainable = True
            transformer.att.query_dense.trainable = True
            transformer.att.key_dense.trainable = True
            transformer.att.value_dense.trainable = True
            transformer.att.combine_heads.trainable = True
            transformer.mlpblock.trainable = True
            transformer.layernorm1.trainable = True
            transformer.layernorm2.trainable = True
            transformer.dropout_layer.trainable = True

    def call(self, x):
        E = self.embedding(x)
        E = self.reshape(E)

        O = self.position_embedding(E)
        for n in range(self.num_layers):
            O, _ = self.transformer[n](O, training=True)
        O = self.transformer_norm(O)

        O = self.meanpooling(O)
        O = self.classification_layer(O)
        return O
    
classification_model=ClassificationModel(ssast_model)
classification_model._set_non_trainable_layers()

test_input = tf.random.normal((32, input_shape[0], input_shape[1], 3))
test_output = classification_model(test_input)
classification_model.summary(show_trainable=True, expand_nested=True)

In [ ]:
classification_model.compile(
    optimizer=keras.optimizers.Adam(1.0),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
    metrics=['accuracy'])

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5,
    patience=3 
)

classification_history = classification_model.fit(
    ds_train_ft.repeat(),
    validation_data=ds_val_ft,
    epochs=80,
    steps_per_epoch=(int(0.1*ds_test_size)//batch_size),
    callbacks=[reduce_lr]
)

In [ ]:
fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('Epochs')
ax1.set_ylabel('loss', color=color)
ax1.plot(classification_history.history['val_loss'], color=color, linestyle='--')
ax1.plot(classification_history.history['loss'], color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

color = 'tab:blue'
ax2.set_ylabel('acc', color=color) 
ax2.plot(classification_history.history['val_accuracy'], color=color, linestyle='--')
ax2.plot(classification_history.history['accuracy'], color=color)
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()
plt.title('Evolution of metrics')
plt.show()

In [ ]:
evaluation = classification_model.evaluate(ds_test_ft)
print("Evaluation accuracy : {:.2f}%".format(evaluation[1]*100))